In [ ]:
from transformers import AutoTokenizer

# Load tokenizer of model that should be fine-tuned
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")

In [ ]:
from spider_ent_data import get_spider_ent_data
spider_ent_data = get_spider_ent_data(tokenizer)

In [ ]:
spider_ent_data
print()

In [ ]:
from spider_data import get_spider_train, get_spider_val

spider_train_data = get_spider_train(tokenizer)
spider_val_data = get_spider_val(tokenizer)

In [ ]:
import re
from transformers import AutoTokenizer
from spider_data import get_spider_val

tok = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base")
data = get_spider_val(tok, 3000)

miss, total = 0, 0
for item in data:
    cand = set()
    for prompt in item["input"]:
        # Regex-Lücken sichtbar machen:
        n_marker = prompt.count("«")
        found = re.findall(r"«\s+(\S+)\s+(\S+)\s*»", prompt)
        if len(found) != n_marker:
            print(f"Regex verliert {n_marker - len(found)} Kandidaten")
        cand |= {(t.lower(), c.lower()) for t, c in found}
    gold = {(t.lower(), c.lower()) for t, cols in item["gold_schema"].items() for c in (cols or [])}
    missing = gold - cand
    miss += len(missing); total += len(gold)
    if missing and miss < 30:
        print("Fehlt:", missing, "| Kandidaten-Beispiel:", sorted(cand)[:3])

print(f"\nStrukturell unerreichbar: {miss}/{total} = {miss/total:.2%}")

In [ ]:
pip install dotenv

In [ ]:
import json
dev = json.load(open("data/spider/dev.json"))  # Pfad ggf. anpassen
bad = [ex for ex in dev if ex["db_id"] == "concert_singer" and " AS T1" in ex["query"].upper()]
ex = bad[0]
print(ex["question"])
print(ex["query"])

In [5]:
from spider_data import get_spider_val, get_spider_train

data = get_spider_train(10000)
for item in data:
    for columns in item["gold_schema"].values():
        if len(columns) == 0:
            print("No columns", item['gold_schema'], item['sql'])

In [2]:
from train import build_training_samples

raw_data = get_spider_train(10000)
samples = build_training_samples(raw_data)
print()

KeyboardInterrupt: 